# BIBM 2026 Tables and Assets

Generate compact, table-centric BIBM evidence assets from existing server-side TriShift result artifacts.


In [ ]:

from pathlib import Path
import json
import math
import shutil
import pandas as pd
import numpy as np

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
OUT = ROOT / "artifacts" / "bibm2026"
TABLE_DIR = OUT / "tables"
ASSET_DIR = OUT / "assets"
for d in (OUT, TABLE_DIR, ASSET_DIR):
    d.mkdir(parents=True, exist_ok=True)

DATASET_LABEL = {
    "adamson": "Adamson",
    "dixit": "Dixit",
    "norman": "Norman",
    "scgen_pbmc_celltype": "PBMC",
    "Adamson": "Adamson",
    "Dixit": "Dixit",
    "Norman": "Norman",
    "PBMC": "PBMC",
}

def clean_model(label):
    if pd.isna(label):
        return label
    s = str(label).strip()
    if s.lower().startswith("trishift"):
        return "TriShift"
    if s.lower() == "biolord":
        return "BioLORD"
    if s.lower() == "genepert":
        return "GenePert"
    if s.lower() == "gears":
        return "GEARS"
    if s.lower() == "scgpt":
        return "scGPT"
    if s.lower() == "cellot":
        return "CellOT"
    return s

def fmt(x, nd=3):
    if x is None or (isinstance(x, float) and math.isnan(x)) or pd.isna(x):
        return "--"
    return f"{float(x):.{nd}f}"

def write_csv_tex(df, stem, caption, label, note=None, column_format=None):
    csv_path = TABLE_DIR / f"{stem}.csv"
    tex_path = TABLE_DIR / f"{stem}.tex"
    df.to_csv(csv_path, index=False)
    if column_format is None:
        column_format = "l" + "c" * (len(df.columns) - 1)
    tex = []
    tex.append("\\begin{table}[t]")
    tex.append("\\centering")
    tex.append("\\caption{" + caption + "}")
    tex.append("\\label{" + label + "}")
    tex.append("\\footnotesize")
    tex.append("\\setlength{\\tabcolsep}{3.2pt}")
    tex.append("\\begin{tabular}{" + column_format + "}")
    tex.append("\\hline")
    tex.append(" & ".join(df.columns) + r" \\")
    tex.append("\\hline")
    for _, row in df.iterrows():
        vals = [str(row[c]) for c in df.columns]
        tex.append(" & ".join(vals) + r" \\")
    tex.append("\\hline")
    tex.append("\\end{tabular}")
    if note:
        tex.append("\\vspace{1mm}")
        tex.append("\\begin{minipage}{0.98\\linewidth}")
        tex.append("\\scriptsize " + note)
        tex.append("\\end{minipage}")
    tex.append("\\end{table}")
    tex_path.write_text("\n".join(tex) + "\n")
    return csv_path, tex_path

main = pd.read_csv(ROOT / "artifacts/paper_figures/main/Fig3_UnseenControlBenchmark/benchmark_summary_all.csv")
main["Dataset"] = main["dataset"].map(DATASET_LABEL)
main["Model"] = main["label"].map(clean_model)
rows = []
for dataset in ["Adamson", "Dixit", "Norman", "PBMC"]:
    sub = main[main["Dataset"] == dataset].copy()
    tri = sub[sub["Model"] == "TriShift"].iloc[0]
    base = sub[sub["Model"] != "TriShift"].copy()
    best_p = base.loc[base["mean_pearson"].idxmax()]
    best_n = base.loc[base["mean_nmse"].idxmin()]
    best_s = base.loc[base["mean_systema_corr_20de_allpert"].idxmax()]
    rows.append({
        "Dataset": dataset,
        "Tri P": fmt(tri["mean_pearson"]),
        "Best P": f"{fmt(best_p['mean_pearson'])} ({best_p['Model']})",
        "Tri nMSE": fmt(tri["mean_nmse"]),
        "Best nMSE": f"{fmt(best_n['mean_nmse'])} ({best_n['Model']})",
        "Tri Sys": fmt(tri["mean_systema_corr_20de_allpert"]),
        "Best Sys": f"{fmt(best_s['mean_systema_corr_20de_allpert'])} ({best_s['Model']})",
    })
table1 = pd.DataFrame(rows)
write_csv_tex(table1, "table1_reference_transfer", "Held-out-control and target-domain reference-transfer summary.", "tab:reference_transfer", "P denotes Pearson correlation; Sys denotes centroid-centered Systema Pearson. For each metric, Best is the strongest non-TriShift baseline available for that dataset and metric.", "lcccccc")

sg = pd.read_csv(ROOT / "artifacts/paper_figures/main/Fig4_NormanGeneralization/fig4_subgroup_summary.csv")
sg["Model"] = sg["label"].map(clean_model)
rows = []
for subgroup in ["single", "seen2", "seen1", "seen0"]:
    sub = sg[sg["subgroup"] == subgroup].copy()
    tri = sub[sub["Model"] == "TriShift"].iloc[0]
    base = sub[sub["Model"] != "TriShift"].copy()
    best_p = base.loc[base["mean_pearson"].idxmax()]
    best_n = base.loc[base["mean_nmse"].idxmin()]
    best_s = base.loc[base["mean_systema_corr_20de_allpert"].idxmax()]
    rows.append({
        "Group": subgroup,
        "Tri P": fmt(tri["mean_pearson"]),
        "Best P": f"{fmt(best_p['mean_pearson'])} ({best_p['Model']})",
        "Tri nMSE": fmt(tri["mean_nmse"]),
        "Best nMSE": f"{fmt(best_n['mean_nmse'])} ({best_n['Model']})",
        "Tri Sys": fmt(tri["mean_systema_corr_20de_allpert"]),
        "Best Sys": f"{fmt(best_s['mean_systema_corr_20de_allpert'])} ({best_s['Model']})",
    })
table2 = pd.DataFrame(rows)
write_csv_tex(table2, "table2_norman_subgroups", "Norman subgroup generalization under held-out-control evaluation.", "tab:norman_subgroups", "seen0 is the hardest subgroup, where neither gene in a test combination appears in training combinations. Best excludes TriShift.", "lcccccc")

ref_sys = pd.read_csv(ROOT / "artifacts/paper_figures/main/Fig3_Ablation/fig3b_reference_systema.csv")
cond_sys = pd.read_csv(ROOT / "artifacts/paper_figures/main/Fig3_Ablation/fig3d_conditioning_systema.csv")
rows = []
for dataset in ["Adamson", "Dixit", "Norman", "PBMC"]:
    r = ref_sys[ref_sys["dataset"] == dataset].set_index("variant")
    c = cond_sys[cond_sys["dataset"] == dataset].set_index("variant")
    rows.append({
        "Dataset": dataset,
        "OT ref": fmt(r.loc["OT", "mean"]),
        "kNN ref": fmt(r.loc["kNN", "mean"]),
        "Full": fmt(c.loc["full", "mean"]),
        "No ref": fmt(c.loc["no reference", "mean"]),
        "No prior": fmt(c.loc["no prior", "mean"]),
    })
table3 = pd.DataFrame(rows)
write_csv_tex(table3, "table3_ablation_systema", "Ablation summary using Systema Pearson.", "tab:ablation", "Higher values indicate stronger centroid-centered response recovery. PBMC has only one stimulus prior, so the no-prior contrast is less diagnostic than no-reference.", "lccccc")

dist = pd.read_csv(ROOT / "artifacts/paper_figures/main/Fig5_DistributionRecovery/fig5_summary_used.csv")
dist["Dataset"] = dist["dataset"].map(DATASET_LABEL)
dist["Model"] = dist["model"].map(clean_model)
rows = []
for dataset in ["Adamson", "Dixit", "Norman", "PBMC"]:
    sub = dist[dist["Dataset"] == dataset].copy()
    tri = sub[sub["Model"] == "TriShift"].iloc[0]
    base = sub[sub["Model"] != "TriShift"].dropna(subset=["scpram_wasserstein_degs_sum", "r2_all_var_mean"])
    best_w = base.loc[base["scpram_wasserstein_degs_sum"].idxmin()]
    best_v = base.loc[base["r2_all_var_mean"].idxmax()]
    rows.append({
        "Dataset": dataset,
        "Tri W": fmt(tri["scpram_wasserstein_degs_sum"]),
        "Best W": f"{fmt(best_w['scpram_wasserstein_degs_sum'])} ({best_w['Model']})",
        "Tri mean r2": fmt(tri["r2_all_mean_mean"]),
        "Tri var r2": fmt(tri["r2_all_var_mean"]),
        "Best var r2": f"{fmt(best_v['r2_all_var_mean'])} ({best_v['Model']})",
    })
table4 = pd.DataFrame(rows)
write_csv_tex(table4, "table4_distribution", "Cell-level distribution recovery on response genes.", "tab:distribution", "W is the summed one-dimensional Wasserstein distance over response genes, where lower is better. Mean r2 and variance r2 are whole-expression squared correlations of predicted and observed distribution statistics, where higher is better.", "lccccc")

deg = pd.read_csv(ROOT / "artifacts/analysis/deg_prediction/deg_prediction_all_summary.csv")
overlap = deg[(deg["dataset"] == "norman") & (deg["metric"] == "overlap_at_20")].copy()
overlap["Model"] = overlap["model"].map(clean_model)
overlap = overlap[["Model", "value"]].sort_values("value", ascending=False)
overlap["Overlap@20"] = overlap["value"].map(fmt)
table5 = overlap[["Model", "Overlap@20"]]
write_csv_tex(table5, "table5_norman_overlap20", "Norman response-gene recovery.", "tab:overlap20", "Overlap@20 measures the intersection between predicted and observed top-20 response genes.", "lc")

fig_candidates = [
    ROOT / "artifacts/paper_figures/main/Fig1_MethodOverview/fig1_framework_dataset_composite_cropped.png",
    ROOT / "artifacts/paper_figures/main/Fig1_MethodOverview/fig1_framework_dataset_composite.svg",
]
asset_records = []
for src in fig_candidates:
    if src.exists():
        dst = ASSET_DIR / src.name
        shutil.copy2(src, dst)
        asset_records.append({"asset": dst.name, "source": str(src)})

manifest = {
    "generated_by": "notebooks/BIBM2026_TablesAndAssets.ipynb",
    "source_tables": {
        "reference_transfer": "artifacts/paper_figures/main/Fig3_UnseenControlBenchmark/benchmark_summary_all.csv",
        "norman_subgroups": "artifacts/paper_figures/main/Fig4_NormanGeneralization/fig4_subgroup_summary.csv",
        "ablation": "artifacts/paper_figures/main/Fig3_Ablation/*.csv",
        "distribution": "artifacts/paper_figures/main/Fig5_DistributionRecovery/fig5_summary_used.csv",
        "overlap20": "artifacts/analysis/deg_prediction/deg_prediction_all_summary.csv",
    },
    "assets": asset_records,
    "tables": sorted(p.name for p in TABLE_DIR.glob("*.csv")),
}
(OUT / "manifest.json").write_text(json.dumps(manifest, indent=2, ensure_ascii=False) + "\n")
print(json.dumps(manifest, indent=2, ensure_ascii=False))
